[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C30_Agent_Harness_Course/04_robustness/04_robustness.ipynb)

# 04 · 鲁棒性（用 MockLLM 注入故障，逐一验证每道防线）

目标：给 agent 加上完整的**防御体系**——**指数退避重试**(区分瞬时/永久错误)、**超时**、**无限循环检测**、**成本追踪 + 预算熔断**、**可观测 trace**，最后组装成一个 **RobustAgent wrapper**，全部用注入故障的 MockLLM + `assert` 验证。

路线：错误分类 → 退避重试(+jitter) → 超时 → 循环检测(指纹) → 成本追踪+预算熔断 → 组装 RobustAgent → ✏️ 练习 → 📖 答案 → 🧪 真实 SDK 重试胶囊。

> 心智模型：**给每类失败配一个应对；给每个可能无限的东西设硬上限(时间/步数/预算)**。纯标准库可跑(超时/重试用确定性模拟，不依赖真实计时)。

## 1 · 错误分类：分清『该重试的』和『重试也没用的』

重试的前提是分类。瞬时错误(429/5xx/超时)重试有望成功；永久错误(400/401)重试永远不会成功、应立刻上报。
先定义错误类型与一个分类器。

In [ ]:
class TransientError(Exception):
    '''瞬时错误：限流、5xx、超时 —— 重试有望成功。'''
    def __init__(self, code, msg=''): super().__init__(msg); self.code = code
class PermanentError(Exception):
    '''永久错误：400 参数非法、401 认证失败 —— 重试无用。'''
    def __init__(self, code, msg=''): super().__init__(msg); self.code = code

def is_retryable(err):
    '''据错误判断是否该重试。'''
    if isinstance(err, TransientError):
        return True
    if isinstance(err, PermanentError):
        return False
    # 也可按 HTTP 状态码: 429/5xx 可重试; 4xx(除429)不可
    code = getattr(err, 'code', None)
    if code is not None:
        return code == 429 or code >= 500
    return False

assert is_retryable(TransientError(429)) is True
assert is_retryable(TransientError(503)) is True
assert is_retryable(PermanentError(400)) is False
assert is_retryable(PermanentError(401)) is False
print('✅ 错误分类就绪：429/5xx 可重试，400/401 不可重试')

## 2 · 退避重试：指数退避 + 抖动(确定性模拟)

对可重试错误，等 `base*2^n + jitter` 再试。这里**不真的 sleep**(用一个记录等待时长的桩，便于断言)。
永久错误立刻抛出、不重试。

In [ ]:
import random

def call_with_retry(fn, max_retries=3, base=1.0, sleep=None, rng=None):
    '''调用 fn()，对可重试错误指数退避重试。sleep 桩记录等待时长(不真睡)。'''
    rng = rng or random.Random(0)
    waits = []
    for attempt in range(max_retries + 1):       # 首次 + 最多 max_retries 次重试
        try:
            return fn(), waits
        except Exception as e:
            if not is_retryable(e):
                raise                            # 永久错误: 立刻上报
            if attempt == max_retries:
                raise                            # 重试耗尽
            wait = base * (2 ** attempt) + rng.uniform(0, base)  # 指数退避 + jitter
            waits.append(wait)
            if sleep: sleep(wait)                 # 真实里 time.sleep(wait)

# 造一个『前2次抛 429、第3次成功』的函数
calls = {'n': 0}
def flaky():
    calls['n'] += 1
    if calls['n'] < 3:
        raise TransientError(429, 'rate limited')
    return 'ok'

result, waits = call_with_retry(flaky, max_retries=3)
print('结果:', result, '| 重试等待序列:', [round(w,2) for w in waits])
assert result == 'ok'
assert calls['n'] == 3, '应在第3次成功'
assert len(waits) == 2, '前2次失败 -> 等待2次'
assert waits[1] > waits[0], '退避时间应指数增长'
# 永久错误不重试
perm_calls = {'n': 0}
def always_400():
    perm_calls['n'] += 1; raise PermanentError(400, 'bad request')
try:
    call_with_retry(always_400, max_retries=3)
except PermanentError:
    pass
assert perm_calls['n'] == 1, '永久错误只试1次、不重试'
print('✅ 退避重试正确：瞬时错误指数退避重试、永久错误立刻上报')

## 3 · 超时：给操作设时限(确定性模拟)

卡死比报错更糟。给操作设时限，超时按失败处理。这里用一个『模拟时钟』：操作声明自己要几个 tick，超过预算即超时。
(真实里用 SDK 的 `timeout=` 参数；这里聚焦超时的**逻辑结构**。)

In [ ]:
class TimeoutError_(Exception):
    pass

def run_with_timeout(op_ticks, budget_ticks):
    '''模拟一个需要 op_ticks 个时钟滴答的操作，预算 budget_ticks。
       op_ticks <= budget_ticks -> 成功(耗时 op_ticks)；否则超时。'''
    elapsed = 0
    for _ in range(op_ticks):
        elapsed += 1
        if elapsed > budget_ticks:
            raise TimeoutError_(f'超时: 用了 {elapsed} tick 仍未完成(预算 {budget_ticks})')
    return f'完成(耗时 {elapsed} tick)'

# 操作要 3 tick、预算 5 -> 成功
print(run_with_timeout(3, 5))
# 操作要 10 tick、预算 5 -> 超时
timed_out = False
try:
    run_with_timeout(10, 5)
except TimeoutError_ as e:
    timed_out = True; print('被拦:', e)
assert timed_out, '超出预算必须超时'
# 超时可被当作可重试错误处理
def slow_op():
    return run_with_timeout(10, 5)
# 把 TimeoutError_ 视为瞬时(可重试)
assert isinstance(TimeoutError_(), Exception)
print('✅ 超时逻辑正确：超出时限按失败处理(可进一步触发重试或降级)')

## 4 · 循环检测：用动作指纹发现原地打转

模型可能反复调同一个 `(工具, 参数)` 毫无进展。给每步算指纹，近窗口内同一指纹重复达阈值就判定死循环、提前止损。

In [ ]:
import json

def make_fingerprint(call):
    '''(工具名, 归一化参数) 作为动作指纹。'''
    return (call['name'], json.dumps(call['input'], sort_keys=True, ensure_ascii=False))

class LoopDetector:
    '''近 window 步内, 同一指纹出现 >= threshold 次 -> 判定死循环。'''
    def __init__(self, window=4, threshold=3):
        self.window, self.threshold = window, threshold
        self.recent = []
    def check(self, call):
        fp = make_fingerprint(call)
        self.recent.append(fp)
        self.recent = self.recent[-self.window:]
        return self.recent.count(fp) >= self.threshold   # True=检测到循环

det = LoopDetector(window=4, threshold=3)
same = {'name':'search','input':{'q':'北京天气'}}
diff = {'name':'search','input':{'q':'上海天气'}}
print('第1次同动作:', det.check(same))   # False
print('第2次同动作:', det.check(same))   # False
print('第3次同动作:', det.check(same))   # True! 触发
assert det.check.__self__.recent.count(make_fingerprint(same)) >= 3
# 不同参数不算重复
det2 = LoopDetector(window=4, threshold=3)
assert det2.check(same) is False
assert det2.check(diff) is False
assert det2.check(same) is False   # same 只出现2次
print('✅ 循环检测正确：同一(工具,参数)近窗口重复达阈值才触发；不同参数不误判')

## 5 · 成本追踪 + 预算熔断

每次 LLM 调用据 `usage` 按单价累加成本；超预算立刻停。建立在模块 03 统一接口的 `usage` 之上。

In [ ]:
def estimate_cost(usage, model='claude-opus-4-8'):
    PRICES = {'claude-opus-4-8':(5.0,25.0), 'claude-sonnet-4-6':(3.0,15.0),
              'claude-haiku-4-5':(1.0,5.0)}   # ($/1M 输入, $/1M 输出)
    pin, pout = PRICES.get(model, (5.0,25.0))
    return usage['input_tokens']/1e6*pin + usage['output_tokens']/1e6*pout

class CostTracker:
    def __init__(self, max_cost, model='claude-opus-4-8'):
        self.max_cost, self.model = max_cost, model
        self.total = 0.0; self.steps = []
    def add(self, usage):
        c = estimate_cost(usage, self.model)
        self.total += c
        self.steps.append(c)
        return self.total
    def over_budget(self):
        return self.total > self.max_cost
    def would_exceed(self, usage):
        '''预测性: 加上这笔会不会超?'''
        return self.total + estimate_cost(usage, self.model) > self.max_cost

ct = CostTracker(max_cost=0.10)
# 每步 2万输入 + 1万输出 token
step_usage = {'input_tokens':20000, 'output_tokens':10000}
per_step = estimate_cost(step_usage)
print(f'每步成本 ≈ ${per_step:.4f}, 预算 ${ct.max_cost}')
n = 0
while not ct.over_budget():
    ct.add(step_usage); n += 1
    if n > 100: break
print(f'跑了 {n} 步后累计 ${ct.total:.4f} 超预算')
assert ct.over_budget()
assert ct.total > 0.10
# 预测性熔断: 下一步会不会超
ct2 = CostTracker(max_cost=per_step * 1.5)
ct2.add(step_usage)
assert ct2.would_exceed(step_usage), '再加一步会超 -> 应提前停'
print('✅ 成本追踪+预算熔断正确：累加成本、超预算停、还能预测性提前熔断')

## 6 · 组装 RobustAgent：四道防线 + trace，包裹朴素循环

把以上全部组装成一个 wrapper，包住模块 03 的循环。用注入故障的 MockLLM 验证**三道保险丝(步数/预算/循环)都生效**、且正常任务能完成。

In [ ]:
class MockLLM:
    def __init__(self, script):
        self.script = list(script); self.calls = 0
    def complete(self, system, messages, tools):
        d = dict(self.script[self.calls]); self.calls += 1
        d.setdefault('text',''); d.setdefault('tool_calls',[])
        d.setdefault('usage',{'input_tokens':1000,'output_tokens':500})
        return d

def dispatch(tools_fns, call):
    fn = tools_fns.get(call['name'])
    if fn is None:
        return {'tool_use_id':call['id'],'content':f'无此工具 {call["name"]}','is_error':True}
    try:
        return {'tool_use_id':call['id'],'content':str(fn(**call['input'])),'is_error':False}
    except Exception as e:
        return {'tool_use_id':call['id'],'content':str(e),'is_error':True}

class RobustAgent:
    def __init__(self, llm, tools_fns, max_steps=15, max_cost=1.0,
                 loop_window=4, loop_threshold=3, model='claude-opus-4-8'):
        self.llm, self.tools_fns = llm, tools_fns
        self.max_steps = max_steps
        self.ct_args = (max_cost, model)
        self.loop_window, self.loop_threshold = loop_window, loop_threshold
    def run(self, system, task):
        history = [{'role':'user','content':task}]
        ct = CostTracker(*self.ct_args)
        det = LoopDetector(self.loop_window, self.loop_threshold)
        trace = []
        for step in range(self.max_steps):                 # 保险丝1: 步数
            d = self.llm.complete(system, history, [])      # (真实里外裹 call_with_retry)
            ct.add(d['usage'])
            trace.append({'step':step, 'stop_reason':d['stop_reason'], 'cost':round(ct.total,4)})
            if ct.over_budget():                            # 保险丝2: 预算
                return {'status':'budget_exceeded','trace':trace,'cost':ct.total}
            history.append({'role':'assistant','text':d['text'],'tool_calls':d['tool_calls']})
            if d['stop_reason'] == 'end_turn':
                return {'status':'done','answer':d['text'],'trace':trace,'cost':ct.total}
            for call in d['tool_calls']:
                if det.check(call):                         # 保险丝3: 循环检测
                    return {'status':'loop_detected','trace':trace,'cost':ct.total}
            results = [dispatch(self.tools_fns, c) for c in d['tool_calls']]  # 错误隔离
            history.append({'role':'user','content':results})
        return {'status':'max_steps','trace':trace,'cost':ct.total}

FNS = {'search': lambda q: f'结果({q})'}
TOOL = lambda q='x': {'id':'t','name':'search','input':{'q':q}}

# A) 正常完成
a = RobustAgent(MockLLM([{'stop_reason':'tool_use','tool_calls':[TOOL('北京')]},
                         {'stop_reason':'end_turn','text':'答完'}]), FNS).run('sys','q')
# B) 死循环 -> loop_detected (反复同一动作)
b = RobustAgent(MockLLM([{'stop_reason':'tool_use','tool_calls':[TOOL('北京')]}]*20), FNS,
                max_steps=20).run('sys','q')
# C) 预算熔断 (每步贵, 预算小)
costly = [{'stop_reason':'tool_use','tool_calls':[TOOL(str(i))],
           'usage':{'input_tokens':500000,'output_tokens':200000}} for i in range(20)]
cc = RobustAgent(MockLLM(costly), FNS, max_steps=20, max_cost=5.0).run('sys','q')
print('A:', a['status'], '| B:', b['status'], '| C:', cc['status'])
assert a['status'] == 'done'
assert b['status'] == 'loop_detected', '反复同一动作应被循环检测拦下(早于 max_steps)'
assert cc['status'] == 'budget_exceeded'
print('✅ RobustAgent 组装成功：正常完成 + 三道保险丝(步数/预算/循环)全部生效, 并产出 trace')

---
## ✏️ 练习 1：实现带 jitter 的退避序列

实现 `backoff_delays(n, base=1.0, cap=60.0, rng=None)`：返回前 `n` 次重试的等待时长列表，每个 = `min(cap, base*2^i) + rng.uniform(0, base)`(指数退避、封顶 cap、加 jitter)。用传入的 `rng` 保证可复现。

In [ ]:
import random
def backoff_delays(n, base=1.0, cap=60.0, rng=None):
    rng = rng or random.Random(0)
    # TODO: 返回长度 n 的列表, 第 i 个 = min(cap, base*2**i) + rng.uniform(0, base)
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
ds = backoff_delays(5, base=1.0, cap=60.0, rng=random.Random(0))
assert len(ds) == 5
# 指数段(未封顶)应递增
assert ds[0] < ds[1] < ds[2], '指数退避应递增'
# 封顶: base*2^i 超过 cap 后, 主项被 cap 限制
big = backoff_delays(10, base=1.0, cap=8.0, rng=random.Random(0))
assert all(d <= 8.0 + 1.0 for d in big), '应被 cap(+最多 base 的 jitter) 限制'
# jitter 使每个都 >= 指数主项
assert ds[0] >= 1.0, '至少是 base*2^0=1'
print('退避序列:', [round(d,2) for d in ds])
print('✅ 练习 1 通过：指数退避 + 封顶 + jitter 正确')

## ✏️ 练习 2：处理输出截断(max_tokens)

若一次调用 `stop_reason=='max_tokens'`，说明输出被截断、不完整。实现 `handle_truncation(decision, retry_fn, max_bumps=2)`：
遇到 `max_tokens` 就调用 `retry_fn()`(它会用更大的 max_tokens 重试)，最多重试 `max_bumps` 次；
拿到非截断结果就返回；重试耗尽仍截断则返回最后那个(带 `'truncated':True`)。

In [ ]:
def handle_truncation(decision, retry_fn, max_bumps=2):
    # TODO: while decision['stop_reason']=='max_tokens' and 还有重试次数:
    #          decision = retry_fn()
    #       若最终仍是 max_tokens, 给 decision 加 'truncated':True
    #       返回 decision
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
# 造一个『前2次截断、第3次完成』的 retry_fn
seq = [{'stop_reason':'max_tokens','text':'半句'},
       {'stop_reason':'max_tokens','text':'还是半句'},
       {'stop_reason':'end_turn','text':'完整答案'}]
state = {'i': 0}
def retry_fn():
    state['i'] += 1; return dict(seq[state['i']])
out = handle_truncation(dict(seq[0]), retry_fn, max_bumps=3)
assert out['stop_reason'] == 'end_turn' and out['text'] == '完整答案'
assert not out.get('truncated')
# 一直截断的情况
state2 = {'i': 0}
always_trunc = lambda: {'stop_reason':'max_tokens','text':'永远半句'}
out2 = handle_truncation({'stop_reason':'max_tokens','text':'x'}, always_trunc, max_bumps=2)
assert out2['stop_reason'] == 'max_tokens' and out2['truncated'] is True
print('✅ 练习 2 通过：截断时加大重试、耗尽则标记 truncated')

## ✏️ 练习 3：给 dispatch 加超时降级

实现 `dispatch_with_timeout(tools_fns, call, budget_ticks, op_ticks_fn)`：
`op_ticks_fn(call)` 返回该工具要几个 tick。若 <= budget 则正常执行(沿用第6节 dispatch)；
若超 budget 则**降级**：返回带 `is_error=True`、content 为 `'工具超时, 已跳过'` 的结果(不崩、让模型换法子)。

In [ ]:
def dispatch_with_timeout(tools_fns, call, budget_ticks, op_ticks_fn):
    # TODO: ticks = op_ticks_fn(call)
    #       ticks > budget_ticks -> 返回超时降级错误(is_error=True)
    #       否则 -> 正常 dispatch(tools_fns, call)
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
fns = {'fast': lambda: 'quick', 'slow': lambda: 'eventually'}
ticks = {'fast': 2, 'slow': 100}
op_ticks = lambda call: ticks[call['name']]
fast_out = dispatch_with_timeout(fns, {'id':'1','name':'fast','input':{}}, 5, op_ticks)
slow_out = dispatch_with_timeout(fns, {'id':'2','name':'slow','input':{}}, 5, op_ticks)
assert fast_out['is_error'] is False and fast_out['content'] == 'quick'
assert slow_out['is_error'] is True and '超时' in slow_out['content']
print('✅ 练习 3 通过：慢工具超时降级成可反馈错误, 快工具正常执行')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案
def backoff_delays(n, base=1.0, cap=60.0, rng=None):
    rng = rng or random.Random(0)
    return [min(cap, base * (2 ** i)) + rng.uniform(0, base) for i in range(n)]

In [ ]:
# 练习 2 参考答案
def handle_truncation(decision, retry_fn, max_bumps=2):
    bumps = 0
    while decision['stop_reason'] == 'max_tokens' and bumps < max_bumps:
        decision = retry_fn()
        bumps += 1
    if decision['stop_reason'] == 'max_tokens':
        decision = dict(decision); decision['truncated'] = True
    return decision

In [ ]:
# 练习 3 参考答案
def dispatch_with_timeout(tools_fns, call, budget_ticks, op_ticks_fn):
    if op_ticks_fn(call) > budget_ticks:
        return {'tool_use_id':call['id'], 'content':'工具超时, 已跳过', 'is_error':True}
    return dispatch(tools_fns, call)

---
## 🧪 真实数据胶囊：真实 Anthropic SDK 的重试与超时

真实里，Anthropic SDK **自带**对 429/5xx 的指数退避重试(`max_retries` 默认 2)和超时(`timeout`)。
你不必手写底层重试，而是**配置**它、并在其上叠 agent 级的循环检测/预算熔断。下面展示配置姿态，并用本课的成本逻辑包一层预算守卫。

> **无 key 也能跑**(成本/预算逻辑纯本地)；有 key + `anthropic` 时真正用带重试的 client。

In [ ]:
import os

def make_robust_client(model='claude-opus-4-8', timeout=30.0, max_retries=4):
    '''返回一个配好超时与重试的真实 client; 无 key/无 SDK 返回 None(调用方回退)。'''
    if os.environ.get('ANTHROPIC_API_KEY'):
        try:
            import anthropic
            # SDK 自带 429/5xx 指数退避重试 + 超时
            return anthropic.Anthropic(timeout=timeout, max_retries=max_retries)
        except ImportError:
            pass
    return None

client = make_robust_client()
print('真实 client:', '已配置(自带重试+超时)' if client else '无(将走本地回退)')
# 无论有无真实 client, agent 级的预算守卫都该工作
ct = CostTracker(max_cost=0.05)
ct.add({'input_tokens':30000, 'output_tokens':15000})
print(f'一步成本 ${ct.total:.4f}, 是否超 $0.05 预算: {ct.over_budget()}')
assert ct.over_budget(), '该步成本应超过 0.05 预算 -> 触发熔断'
print('✅ 胶囊: SDK 自带底层重试/超时, 我们在其上叠 agent 级预算熔断(无 key 也成立)')

**🧪 胶囊练习**：实现 `safe_complete(client, params, mock_decision)`：有真实 `client` 就调 `client.messages.create(**params)` 并返回响应；**任何异常(限流耗尽/超时/网络)都回退**返回 `mock_decision`。**绝不让一次真实调用失败崩掉整个流程**——这是『真实增强、mock 保底』的最后一道防线。

In [ ]:
def safe_complete(client, params, mock_decision):
    # TODO: client 为 None -> 直接返回 mock_decision
    #       否则 try: 调用 client.messages.create(**params), 返回响应
    #            except Exception: 返回 mock_decision (绝不抛出)
    raise NotImplementedError

In [ ]:
# 自测: 无 client 必回退; 有 client 但调用失败也回退
mock_dec = {'stop_reason':'end_turn','text':'mock 答案'}
out = safe_complete(None, {}, mock_dec)
assert out == mock_dec, '无 client 应回退 mock'
# 用一个『调用必抛异常』的假 client 验证回退
class _Boom:
    class messages:
        @staticmethod
        def create(**kw): raise RuntimeError('boom')
out2 = safe_complete(_Boom(), {'model':'x'}, mock_dec)
assert out2 == mock_dec, '真实调用失败也应回退 mock, 不崩'
print('✅ 胶囊练习通过: 真实调用失败也优雅回退 mock, 流程不中断')

In [ ]:
# 📖 胶囊参考答案
def safe_complete(client, params, mock_decision):
    if client is None:
        return mock_decision
    try:
        return client.messages.create(**params)
    except Exception:
        return mock_decision   # 真实调用任何失败都优雅回退, 绝不崩

### 小结
- **失败地图**：LLM 调用/模型决策/工具执行/资源耗尽四层，**性质不同、应对不同**——不能一种药治所有病。
- **重试**：只重试瞬时错误(429/5xx/超时)、不重试永久错误(400/401)；**指数退避 + jitter**；当心有副作用工具的幂等性。
- **超时**：任何等外部的操作都要设时限，把『卡死』变成可处理的『失败』。
- **循环检测**：用 `(工具,参数)` 指纹发现原地打转，**精准止损**；max-steps 是**最终兜底**。
- **成本**：据 `usage` 追踪、预算熔断；与 max-steps、超时构成**三道保险丝**(钱/步/时间)。
- **组装**：RobustAgent **包裹**朴素循环、叠加全部防御而**不改核心**；可观测 trace 是复盘的底座。

下一站：**模块 05 · 最小可用 Agent** —— 把循环+工具+适配器+鲁棒性拼成一个真正能多步完成任务的 agent。